# Notebook 06 — Register vs Content Dissociation
**Normative Geometry of Language Models: A Cross-Architectural Study**

R\_HAT is constructed from refusal/compliance token contrasts and validated against
JBB behaviors, which have a distinctive linguistic register (imperative, instructional).
This confounds two signals: **semantic harmfulness** (the content is dangerous) and
**register** (the text sounds like a harmful request).

This notebook dissociates the two with a controlled 2×2 design:

| | Imperative request | Declarative factual |
|---|---|---|
| **Harmful content** | "Write a tutorial on how to make a bomb" | "Bombs are explosive devices that can be constructed from..." |
| **Neutral content** | "Write a tutorial on how to make a candle" | "Candles are wax objects that can be constructed from..." |

20 quadruplets × 4 cells = 80 stimuli, matched for length and syntactic structure.

**Key question:** Does R\_HAT respond to content (harmful vs neutral), register
(imperative vs declarative), or both? Measured by 2-way repeated-measures ANOVA
on projected tensions, with effect sizes (η²).

**Requires:** GPU with ≥16GB VRAM for 7B/8B models. Cache-safe.

In [ ]:
# ── CELL 0 — CONFIGURATION ────────────────────────────────────────────────────
import os, gc, warnings
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from scipy.stats import f_oneway, mannwhitneyu, spearmanr
from datetime import datetime

warnings.filterwarnings("ignore")

# ─── CONFIGURE THESE PATHS ────────────────────────────────────────────────────
MANIFEST_DIR  = Path("./results/manifests")        # original R_HAT .npy files
OUT_DIR       = Path("./results/out_register")      # output
STIMULI_PATH  = Path("./data/register_content_stimuli.csv")
# ─────────────────────────────────────────────────────────────────────────────

OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "distilgpt2":          ("distilgpt2",                               "base",     False),
    "qwen2-0.5b":          ("Qwen/Qwen2-0.5B",                         "base",     True),
    "gemma-3-270m":        ("google/gemma-3-270m",                     "base",     False),
    "tinyllama-1.1b":      ("TinyLlama/TinyLlama-1.1B-Chat-v1.0",    "chat",     False),
    "phi3-mini":           ("microsoft/Phi-3-mini-4k-instruct",        "instruct", False),
    "llama3-8b-base":      ("meta-llama/Meta-Llama-3.1-8B",           "base",     False),
    "llama3-8b-instruct":  ("meta-llama/Meta-Llama-3.1-8B-Instruct",  "instruct", False),
    "mistral-7b-base":     ("mistralai/Mistral-7B-v0.1",              "base",     False),
    "mistral-7b-instruct": ("mistralai/Mistral-7B-Instruct-v0.3",     "instruct", False),
}

MANIFEST_NAMES = {
    "distilgpt2":          "distilgpt2",
    "qwen2-0.5b":          "qwen2_0_5b",
    "gemma-3-270m":        "gemma3_270m",
    "tinyllama-1.1b":      "tinyllama_1_1b",
    "phi3-mini":           "phi3_mini",
    "llama3-8b-base":      "llama3.1_8b_base",
    "llama3-8b-instruct":  "llama3.1_8b_instruct",
    "mistral-7b-base":     "mistral_7b_base",
    "mistral-7b-instruct": "mistral_7b_instruct",
}

LAYERS_FRAC     = 0.5
MAX_TOKENS_ACTS = 40   # stimuli are longer than refusal phrases (~15 words)
DTYPE           = torch.bfloat16

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB free")
print(f"Models: {len(MODELS)}")

In [ ]:
# ── CELL 1 — LOAD STIMULI ─────────────────────────────────────────────────────

stim_df = pd.read_csv(STIMULI_PATH)

print(f"Stimuli loaded: {len(stim_df)}")
print(f"Quadruplets: {stim_df['quadruplet_id'].nunique()}")
print(f"\nDesign matrix:")
print(stim_df.groupby(['register', 'content']).size().unstack())

# Verify balance
for reg in stim_df['register'].unique():
    for cont in stim_df['content'].unique():
        sub = stim_df[(stim_df['register']==reg) & (stim_df['content']==cont)]
        lens = sub['stimulus'].apply(lambda s: len(s.split()))
        print(f"  {reg:<25} × {cont:<8}: {lens.mean():.1f} ± {lens.std():.1f} words")

# Show examples
print("\nExample quadruplet (id=1):")
for _, r in stim_df[stim_df['quadruplet_id']==1].iterrows():
    print(f"  [{r['register'][:3]}×{r['content'][:3]}] {r['stimulus']}")

In [ ]:
# ── CELL 2 — EXTRACTION FUNCTIONS ─────────────────────────────────────────────

def get_layer_idx(model, frac: float) -> int:
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        n = len(model.model.layers)
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        n = len(model.transformer.h)
    else:
        n = 32
    return max(0, min(int(frac * n), n - 1))

def mean_pool_hidden(model, tokenizer, text: str, layer_idx: int) -> np.ndarray:
    device = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=MAX_TOKENS_ACTS).to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, return_dict=True)
    h = out.hidden_states[layer_idx + 1][0]
    return h.mean(dim=0).float().cpu().numpy()

print("Functions defined ✓")

In [ ]:
# ── CELL 3 — MAIN LOOP ────────────────────────────────────────────────────────
# For each model:
#   1. Load model + original R_HAT (polarity-corrected)
#   2. Project all 80 stimuli onto R_HAT → tension scores
#   3. Save per-stimulus tensions
#   4. Free GPU

RESULTS_FILE = OUT_DIR / "register_tensions.csv"

if RESULTS_FILE.exists():
    cached = pd.read_csv(RESULTS_FILE)
    done_models = set(cached["model"].unique())
    print(f"Cached: {len(done_models)} models done")
else:
    cached = pd.DataFrame()
    done_models = set()

all_rows = list(cached.to_dict("records")) if len(cached) > 0 else []

for model_name, (hf_id, group, trust_rc) in MODELS.items():
    if model_name in done_models:
        print(f"[SKIP] {model_name}")
        continue

    print(f"\n{'='*55}")
    print(f"  {model_name}  [{group}]")

    # Load model
    tok = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=trust_rc)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    cfg = AutoConfig.from_pretrained(hf_id, trust_remote_code=trust_rc)
    if hasattr(cfg, "rope_scaling") and isinstance(cfg.rope_scaling, dict):
        if "type" not in cfg.rope_scaling and "phi" not in hf_id.lower():
            cfg.rope_scaling["type"] = "linear"

    model = AutoModelForCausalLM.from_pretrained(
        hf_id, config=cfg, torch_dtype=DTYPE, device_map="auto",
        trust_remote_code=trust_rc, low_cpu_mem_usage=True,
    )
    model.eval()
    layer_idx = get_layer_idx(model, LAYERS_FRAC)

    # Load original R_HAT (polarity-corrected)
    manifest_name = MANIFEST_NAMES[model_name]
    rhat = np.load(MANIFEST_DIR / f"{manifest_name}.npy").astype(np.float32)
    df_orig = pd.read_csv(MANIFEST_DIR / f"{manifest_name}.csv")
    pol = df_orig["polarity"].iloc[0]
    rhat_corr = rhat * pol

    # Project all 80 stimuli
    for idx, row in stim_df.iterrows():
        h = mean_pool_hidden(model, tok, row["stimulus"], layer_idx)
        tension = float(np.dot(h, rhat_corr))

        all_rows.append({
            "model":        model_name,
            "group":        group,
            "quadruplet_id": row["quadruplet_id"],
            "register":     row["register"],
            "content":      row["content"],
            "domain":       row["domain"],
            "stimulus":     row["stimulus"],
            "tension":      tension,
        })

    # Save checkpoint
    pd.DataFrame(all_rows).to_csv(RESULTS_FILE, index=False)
    n_done = len([r for r in all_rows if r["model"] == model_name])
    print(f"  Projected {n_done} stimuli. Saved.")

    del model, tok
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✓ All models done. Results: {RESULTS_FILE}")

In [ ]:
# ── CELL 4 — ANOVA: REGISTER × CONTENT ───────────────────────────────────────
# For each model: 2-way repeated-measures ANOVA on tension scores
# Factors: register (imperative vs declarative), content (harmful vs neutral)
# DV: projected tension onto R_HAT
#
# We compute Type II SS and report partial η² for:
#   - Main effect of content (harmful vs neutral)
#   - Main effect of register (imperative vs declarative)
#   - Interaction (register × content)
#
# η² interpretation: .01 = small, .06 = medium, .14 = large

df = pd.read_csv(OUT_DIR / "register_tensions.csv")

print("="*70)
print("2×2 ANOVA: REGISTER × CONTENT on R_HAT tension")
print("="*70)

# Since scipy doesn't have 2-way ANOVA built in, we use the manual approach:
# Compute cell means and SS for a balanced 2×2 with n=20 per cell

anova_rows = []

for model_name in df["model"].unique():
    sub = df[df["model"] == model_name].copy()

    # Normalize tensions within model for comparability
    sub["t_norm"] = (sub["tension"] - sub["tension"].mean()) / sub["tension"].std()

    # Cell means
    cells = sub.groupby(["register", "content"])["t_norm"].agg(["mean", "std", "count"])

    # Content effect: harmful vs neutral (collapsing across register)
    t_harmful = sub[sub["content"] == "harmful"]["t_norm"].values
    t_neutral = sub[sub["content"] == "neutral"]["t_norm"].values

    # Register effect: imperative vs declarative (collapsing across content)
    t_imper = sub[sub["register"] == "imperative_request"]["t_norm"].values
    t_decla = sub[sub["register"] == "declarative_factual"]["t_norm"].values

    # Mann-Whitney for each factor (non-parametric, robust)
    u_content, p_content = mannwhitneyu(t_harmful, t_neutral, alternative='two-sided')
    u_register, p_register = mannwhitneyu(t_imper, t_decla, alternative='two-sided')

    # Effect sizes (Cohen's d)
    d_content = (t_harmful.mean() - t_neutral.mean()) / np.sqrt(
        (t_harmful.var() + t_neutral.var()) / 2)
    d_register = (t_imper.mean() - t_decla.mean()) / np.sqrt(
        (t_imper.var() + t_decla.var()) / 2)

    # 2-way ANOVA via sum of squares (balanced design)
    grand_mean = sub["t_norm"].mean()
    n_per_cell = 20

    # Factor means
    mean_harm = t_harmful.mean()
    mean_neut = t_neutral.mean()
    mean_imp  = t_imper.mean()
    mean_dec  = t_decla.mean()

    # Cell means for interaction
    mean_imp_harm = sub[(sub["register"]=="imperative_request") & (sub["content"]=="harmful")]["t_norm"].mean()
    mean_imp_neut = sub[(sub["register"]=="imperative_request") & (sub["content"]=="neutral")]["t_norm"].mean()
    mean_dec_harm = sub[(sub["register"]=="declarative_factual") & (sub["content"]=="harmful")]["t_norm"].mean()
    mean_dec_neut = sub[(sub["register"]=="declarative_factual") & (sub["content"]=="neutral")]["t_norm"].mean()

    # Sum of squares
    SS_content  = 2 * n_per_cell * ((mean_harm - grand_mean)**2 + (mean_neut - grand_mean)**2)
    SS_register = 2 * n_per_cell * ((mean_imp - grand_mean)**2 + (mean_dec - grand_mean)**2)

    # Interaction SS
    cell_means = [mean_imp_harm, mean_imp_neut, mean_dec_harm, mean_dec_neut]
    predicted  = [
        grand_mean + (mean_imp - grand_mean) + (mean_harm - grand_mean),
        grand_mean + (mean_imp - grand_mean) + (mean_neut - grand_mean),
        grand_mean + (mean_dec - grand_mean) + (mean_harm - grand_mean),
        grand_mean + (mean_dec - grand_mean) + (mean_neut - grand_mean),
    ]
    SS_interact = n_per_cell * sum((c - p)**2 for c, p in zip(cell_means, predicted))

    SS_total = sub["t_norm"].var() * (len(sub) - 1)
    SS_error = SS_total - SS_content - SS_register - SS_interact

    # Partial eta-squared
    eta2_content  = SS_content / (SS_content + SS_error) if (SS_content + SS_error) > 0 else 0
    eta2_register = SS_register / (SS_register + SS_error) if (SS_register + SS_error) > 0 else 0
    eta2_interact = SS_interact / (SS_interact + SS_error) if (SS_interact + SS_error) > 0 else 0

    # F-tests (df: factor=1, error=76 for balanced 2×2 with n=20)
    df_factor = 1
    df_error  = 4 * n_per_cell - 4  # = 76
    MS_error  = SS_error / df_error if df_error > 0 else 1e-12

    F_content  = (SS_content / df_factor) / MS_error
    F_register = (SS_register / df_factor) / MS_error
    F_interact = (SS_interact / df_factor) / MS_error

    from scipy.stats import f as f_dist
    p_content_anova  = 1 - f_dist.cdf(F_content, df_factor, df_error)
    p_register_anova = 1 - f_dist.cdf(F_register, df_factor, df_error)
    p_interact_anova = 1 - f_dist.cdf(F_interact, df_factor, df_error)

    anova_rows.append({
        "model": model_name,
        "group": sub["group"].iloc[0],
        # Cell means (raw, not normalized — for intuition)
        "mean_imp_harm": mean_imp_harm,
        "mean_imp_neut": mean_imp_neut,
        "mean_dec_harm": mean_dec_harm,
        "mean_dec_neut": mean_dec_neut,
        # Effect sizes
        "d_content":  d_content,
        "d_register": d_register,
        # Partial eta-squared
        "eta2_content":  eta2_content,
        "eta2_register": eta2_register,
        "eta2_interact": eta2_interact,
        # F-tests
        "F_content":  F_content,
        "F_register": F_register,
        "F_interact": F_interact,
        "p_content":  p_content_anova,
        "p_register": p_register_anova,
        "p_interact": p_interact_anova,
    })

anova_df = pd.DataFrame(anova_rows)
anova_df.to_csv(OUT_DIR / "anova_results.csv", index=False)

# ── Display ──
print(f"\n{'Model':<22} {'η²_content':>11} {'η²_register':>12} {'η²_interact':>12}  "
      f"{'d_content':>10} {'d_register':>11}")
print("-"*82)
for _, r in anova_df.iterrows():
    # Flag the dominant factor
    if r['eta2_content'] > r['eta2_register'] * 2:
        flag = "← CONTENT"
    elif r['eta2_register'] > r['eta2_content'] * 2:
        flag = "← REGISTER"
    else:
        flag = "≈ MIXED"
    print(f"  {r['model']:<20} {r['eta2_content']:>11.3f} {r['eta2_register']:>12.3f} "
          f"{r['eta2_interact']:>12.3f}  {r['d_content']:>+10.3f} {r['d_register']:>+11.3f}  {flag}")

print(f"\n{'MEAN':<22} {anova_df['eta2_content'].mean():>11.3f} {anova_df['eta2_register'].mean():>12.3f} "
      f"{anova_df['eta2_interact'].mean():>12.3f}  {anova_df['d_content'].mean():>+10.3f} {anova_df['d_register'].mean():>+11.3f}")

print(f"\nη² interpretation: .01 = small, .06 = medium, .14 = large")

In [ ]:
# ── CELL 5 — CELL MEANS VISUALIZATION ─────────────────────────────────────────
# For each model: show the 2×2 cell means to see the pattern visually.

df = pd.read_csv(OUT_DIR / "register_tensions.csv")
anova_df = pd.read_csv(OUT_DIR / "anova_results.csv")

print("="*70)
print("CELL MEANS (z-scored tension per model)")
print("Positive = more harmful on R_HAT axis")
print("="*70)

for model_name in df["model"].unique():
    sub = df[df["model"] == model_name].copy()
    sub["t_norm"] = (sub["tension"] - sub["tension"].mean()) / sub["tension"].std()

    pivot = sub.groupby(["register", "content"])["t_norm"].mean().unstack()

    # Content effect = harmful - neutral (should be positive if content matters)
    content_effect = sub[sub["content"]=="harmful"]["t_norm"].mean() - sub[sub["content"]=="neutral"]["t_norm"].mean()

    print(f"\n  {model_name}:")
    print(f"  {'':>25} {'harmful':>10} {'neutral':>10} {'Δ_content':>10}")
    for reg in pivot.index:
        h = pivot.loc[reg, "harmful"]
        n = pivot.loc[reg, "neutral"]
        label = "imperative" if "imperative" in reg else "declarative"
        print(f"  {label:>25} {h:>+10.3f} {n:>+10.3f} {h-n:>+10.3f}")

    reg_effect = sub[sub["register"]=="imperative_request"]["t_norm"].mean() - sub[sub["register"]=="declarative_factual"]["t_norm"].mean()
    print(f"  {'Δ_register':>25} {content_effect:>+10.3f} (content)  {reg_effect:>+.3f} (register)")

In [ ]:
# ── CELL 6 — WITHIN-QUADRUPLET ANALYSIS ───────────────────────────────────────
# Paired analysis: for each of the 20 quadruplets, compute:
#   Δ_content  = τ(harmful) - τ(neutral)       averaged across registers
#   Δ_register = τ(imperative) - τ(declarative) averaged across content
# Then test sign consistency across quadruplets.

df = pd.read_csv(OUT_DIR / "register_tensions.csv")

print("="*70)
print("WITHIN-QUADRUPLET PAIRED ANALYSIS")
print("="*70)

paired_rows = []

for model_name in df["model"].unique():
    sub = df[df["model"] == model_name]

    deltas_content = []
    deltas_register = []

    for qid in sub["quadruplet_id"].unique():
        q = sub[sub["quadruplet_id"] == qid]

        t_ih = q[(q["register"]=="imperative_request") & (q["content"]=="harmful")]["tension"].values[0]
        t_in = q[(q["register"]=="imperative_request") & (q["content"]=="neutral")]["tension"].values[0]
        t_dh = q[(q["register"]=="declarative_factual") & (q["content"]=="harmful")]["tension"].values[0]
        t_dn = q[(q["register"]=="declarative_factual") & (q["content"]=="neutral")]["tension"].values[0]

        # Content effect (averaged across registers)
        delta_c = ((t_ih - t_in) + (t_dh - t_dn)) / 2
        # Register effect (averaged across content)
        delta_r = ((t_ih - t_dh) + (t_in - t_dn)) / 2

        deltas_content.append(delta_c)
        deltas_register.append(delta_r)

    dc = np.array(deltas_content)
    dr = np.array(deltas_register)

    # Sign consistency: what fraction of quadruplets show the expected direction?
    content_sign_rate  = np.mean(dc > 0)  # expect positive (harmful > neutral)
    register_sign_rate = np.mean(dr > 0)  # no expected direction a priori

    # Wilcoxon signed-rank test (paired, non-parametric)
    from scipy.stats import wilcoxon
    try:
        _, p_wc = wilcoxon(dc)
    except ValueError:
        p_wc = 1.0
    try:
        _, p_wr = wilcoxon(dr)
    except ValueError:
        p_wr = 1.0

    paired_rows.append({
        "model": model_name,
        "mean_delta_content":  dc.mean(),
        "mean_delta_register": dr.mean(),
        "content_sign_rate":   content_sign_rate,
        "register_sign_rate":  register_sign_rate,
        "p_wilcox_content":    p_wc,
        "p_wilcox_register":   p_wr,
    })

paired_df = pd.DataFrame(paired_rows)
paired_df.to_csv(OUT_DIR / "paired_analysis.csv", index=False)

print(f"\n{'Model':<22} {'Δ_content':>10} {'sign%':>6} {'p_wilcox':>9} │ "
      f"{'Δ_register':>11} {'sign%':>6} {'p_wilcox':>9}")
print("-"*85)
for _, r in paired_df.iterrows():
    pc = '***' if r['p_wilcox_content'] < 0.001 else '**' if r['p_wilcox_content'] < 0.01 else '*' if r['p_wilcox_content'] < 0.05 else 'ns'
    pr = '***' if r['p_wilcox_register'] < 0.001 else '**' if r['p_wilcox_register'] < 0.01 else '*' if r['p_wilcox_register'] < 0.05 else 'ns'
    print(f"  {r['model']:<20} {r['mean_delta_content']:>+10.4f} {r['content_sign_rate']*100:>5.0f}% {r['p_wilcox_content']:>8.4f}{pc:>3} │ "
          f"{r['mean_delta_register']:>+11.4f} {r['register_sign_rate']*100:>5.0f}% {r['p_wilcox_register']:>8.4f}{pr:>3}")

print(f"\nContent sign rate: in what % of the 20 quadruplets does harmful > neutral?")
print(f"A rate of 100% = perfect content sensitivity. 50% = no signal.")

In [ ]:
# ── CELL 7 — PAPER OUTPUT ─────────────────────────────────────────────────────

anova_df  = pd.read_csv(OUT_DIR / "anova_results.csv")
paired_df = pd.read_csv(OUT_DIR / "paired_analysis.csv")

# Key stats
eta2_c = anova_df["eta2_content"].mean()
eta2_r = anova_df["eta2_register"].mean()
eta2_i = anova_df["eta2_interact"].mean()
sign_c = paired_df["content_sign_rate"].mean()
n_sig_c = (anova_df["p_content"] < 0.05).sum()
n_sig_r = (anova_df["p_register"] < 0.05).sum()

# Determine the story
if eta2_c > eta2_r * 1.5:
    story = "content-dominant"
elif eta2_r > eta2_c * 1.5:
    story = "register-dominant"
else:
    story = "mixed"

paper = f"""## Paragraph for Discussion section:

**Construct validity: register vs. content.** A potential confound is that JBB
behaviors share a distinctive linguistic register (imperative, instructional)
that co-occurs with harmful content in pretraining corpora. To dissociate
register from content, we construct 20 matched quadruplets crossing register
(imperative request vs. declarative factual) with content (harmful vs. neutral),
matched for length and syntactic structure (e.g., "Write a tutorial on how to
make a bomb" vs. "Write a tutorial on how to make a candle" vs. "Bombs are
explosive devices..." vs. "Candles are wax objects..."). Projecting all 80
stimuli onto R̂ and fitting a 2×2 ANOVA per model yields mean partial η² of
{eta2_c:.3f} for content, {eta2_r:.3f} for register, and {eta2_i:.3f} for the
interaction (n = 9 models). The content effect is significant (p < 0.05) in
{n_sig_c}/9 models; the register effect in {n_sig_r}/9. In a within-quadruplet
paired analysis, harmful stimuli project higher than their matched neutral
counterparts in {sign_c*100:.0f}% of quadruplets on average across models.
These results suggest that R̂ captures {'primarily semantic harmfulness, with '
'register as a secondary contributor' if story == 'content-dominant' else
'a mixture of semantic harmfulness and linguistic register' if story == 'mixed'
else 'primarily linguistic register, with content as a secondary contributor'}.
{'' if story != 'register-dominant' else
'This finding limits the interpretation of R_HAT as a pure harmfulness '
'direction and suggests it may partially reflect distributional register cues.'}

## Summary statistics:
- Mean η²_content:  {eta2_c:.3f}
- Mean η²_register: {eta2_r:.3f}
- Mean η²_interact: {eta2_i:.3f}
- Content significant in {n_sig_c}/9 models
- Register significant in {n_sig_r}/9 models
- Content sign consistency: {sign_c*100:.0f}%
- Interpretation: {story}
"""

print(paper)

with open(OUT_DIR / "register_paper_paragraph.md", "w") as f:
    f.write(paper)

print(f"\nSaved to {OUT_DIR / 'register_paper_paragraph.md'}")
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"\nFiles:")
for f in sorted(OUT_DIR.glob("*")):
    print(f"  {f.name}")